# Phase 8b — Part A: the tip as a critical *chain-of-thought* length (thinking ON)

The RESULTS.md run had `enable_thinking=False`, so the think block was a constant
~2-token scaffold — the CoT length could not act as an order parameter (the phase-8
degeneracy guard correctly refused to fit an `L*`). This notebook fixes that by
**regenerating** the language-switching prompts with **`enable_thinking=True`**, so the
model produces think blocks of varying length, and tests the hypothesis directly on
the text:

> a completion tips into the weird behaviour (switches language) when its think
> phase is too short — it *falls below a critical length* `L*`.

**No GPU, no draft, no DeepSpec.** Just generation on OpenRouter + text analysis.
The "tip" is detected as a **non-Latin script onset** in the answer (Cyrillic / CJK /
Arabic / …, not only Chinese — the last run's strongest case was actually *Russian*).

Needs: an **OpenRouter** key (generation) and an **HF token** (to load the WeirdChat
prompts + the Qwen tokenizer). Both are read from Colab secrets if present.


In [ ]:
# === Cell 1 — config =========================================================
import os
# Pull keys from Colab secrets (Runtime > Secrets) if available, else use env.
try:
    from google.colab import userdata
    for k in ("OPENROUTER_API_KEY","HF_TOKEN"):
        v=None
        try: v=userdata.get(k)
        except Exception: v=None
        if v: os.environ.setdefault(k,v)
except Exception as e:
    print("colab secrets unavailable:", e)

MODEL        = "qwen/qwen3.6-35b-a3b"          # subject model on OpenRouter (on-policy)
BEHAVIORS    = ["language-switching-english"]  # the behaviour whose tip we localize
PATTERNS     = 6      # patterns per behaviour (highest OpenRouter reproduction first)
PROMPTS      = 8      # prompts per pattern
SAMPLES      = 64     # completions per prompt — the main knob; 16 for a quick look
TEMPERATURE  = 1.0    # WeirdChat protocol
MAX_TOKENS   = 1024
CONCURRENCY  = 24
# How to ask for thinking. OpenRouter's unified flag; if no reasoning comes back,
# try {"reasoning": {"effort": "high"}} or a provider that exposes reasoning.
THINK_EXTRA_BODY = {"reasoning": {"enabled": True}}
FOREIGN_RUN  = 3      # a "switch" = a run of >=3 non-Latin letters in the answer
BRANCH       = "claude/repo-published-weights-u71yew"
OUTPUT       = "/content/drive/MyDrive/weirdspec/thinking_gen.jsonl"   # resumable
MOUNT_DRIVE  = True

assert os.environ.get("OPENROUTER_API_KEY"), "set OPENROUTER_API_KEY (Colab secret or env)"
print(f"plan: {len(BEHAVIORS)}x{PATTERNS}x{PROMPTS}x{SAMPLES} = "
      f"{len(BEHAVIORS)*PATTERNS*PROMPTS*SAMPLES} completions, T={TEMPERATURE}, thinking ON")


In [ ]:
# === Cell 2 — mount Drive + install weirdchat (no GPU, no torch) ============
import os, sys, subprocess, importlib
if MOUNT_DRIVE:
    try:
        from google.colab import drive; drive.mount("/content/drive")
    except Exception as e:
        print("drive mount skipped:", e)

# Clone once (public repo). Hardcode the branch to dodge any interpolation surprise.
if not os.path.exists("/content/WeirdChat"):
    subprocess.run(["git","clone","--depth","1","--branch",
                    "claude/repo-published-weights-u71yew",
                    "https://github.com/Erikiss/WeirdChat","/content/WeirdChat"], check=True)
# Editable install pulls weirdchat's LIGHT deps (openai, docent, huggingface_hub…;
# no torch). check=True so a failure is LOUD instead of silently swallowed.
subprocess.run([sys.executable,"-m","pip","install","-q","-e","/content/WeirdChat","nest_asyncio"],
               check=True)
# Make it importable in THIS kernel with no restart: an editable-install .pth is
# not reprocessed mid-session, so put the repo root on sys.path directly.
if "/content/WeirdChat" not in sys.path:
    sys.path.insert(0, "/content/WeirdChat")
importlib.invalidate_caches()
import weirdchat as _wc
print("weirdchat OK:", os.path.dirname(_wc.__file__))


In [ ]:
# === Cell 3 — select the language-switching prompts =========================
import weirdchat as wc
sel=[]   # (behavior, pattern_id, prompt_object)
for beh in BEHAVIORS:
    pats = wc.patterns(behavior_id=beh, subject_model=MODEL)
    pats.sort(key=lambda p:(p.openrouter_replication.rate if p.openrouter_replication else 0) or 0,
              reverse=True)
    for p in pats[:PATTERNS]:
        for pr in wc.prompts(p)[:PROMPTS]:
            sel.append((beh, p.pattern_id, pr))
print(f"{len(sel)} prompt cells; {len(sel)*SAMPLES} completions planned")
if sel:
    print("top pattern reproduction rates:",
          [round((p.openrouter_replication.rate if p.openrouter_replication else 0) or 0,2)
           for p in pats[:PATTERNS]])


In [ ]:
# === Cell 4 — generate (async, resumable) ===================================
import json, os, re, hashlib, asyncio
from openai import AsyncOpenAI

def read_jsonl(path):
    rows=[]
    if os.path.exists(path):
        with open(path,encoding="utf-8") as f:
            for line in f:
                line=line.strip()
                if line: rows.append(json.loads(line))
    return rows
def prompt_key(pr):
    h=hashlib.sha1("||".join(m.role+":"+m.content for m in pr.messages).encode()).hexdigest()
    return h[:12]
def get_reasoning(msg):
    """OpenRouter returns reasoning in message.reasoning / reasoning_details / model_extra."""
    r=getattr(msg,"reasoning",None)
    me=getattr(msg,"model_extra",None) or {}
    if r is None: r=me.get("reasoning")
    if r is None:
        det=me.get("reasoning_details") or getattr(msg,"reasoning_details",None)
        if det:
            try: r="".join(d.get("text","") for d in det if isinstance(d,dict))
            except Exception: r=None
    return r
def split_think(msg):
    """(reasoning_text, answer_text, mode)."""
    content=msg.content or ""
    r=get_reasoning(msg)
    if r: return r, content, "reasoning_field"
    m=re.search(r"<think>(.*?)</think>(.*)", content, re.DOTALL)
    if m: return m.group(1), m.group(2), "inline_think"
    return "", content, "no_think"

client=AsyncOpenAI(base_url="https://openrouter.ai/api/v1",
                   api_key=os.environ["OPENROUTER_API_KEY"], timeout=180, max_retries=3)
done={(r["pattern_id"], r["prompt_key"], r["sample_idx"]) for r in read_jsonl(OUTPUT)}
os.makedirs(os.path.dirname(OUTPUT) or ".", exist_ok=True)
_outf=open(OUTPUT,"a",encoding="utf-8"); _lock=asyncio.Lock(); _sem=asyncio.Semaphore(CONCURRENCY)

async def one(beh,pid,pr,k):
    pk=prompt_key(pr)
    if (pid,pk,k) in done: return
    async with _sem:
        try:
            comp=await client.chat.completions.create(
                model=MODEL,
                messages=[{"role":m.role,"content":m.content} for m in pr.messages],
                temperature=TEMPERATURE, max_tokens=MAX_TOKENS, extra_body=THINK_EXTRA_BODY)
        except Exception as e:
            print("  gen error:", type(e).__name__); return
    reasoning,answer,mode=split_think(comp.choices[0].message)
    rec=dict(behavior=beh, pattern_id=pid, prompt_key=pk, sample_idx=k, mode=mode,
             reasoning=reasoning[:6000], answer=answer[:6000],
             reasoning_chars=len(reasoning), answer_chars=len(answer))
    async with _lock:
        _outf.write(json.dumps(rec,ensure_ascii=False)+"\n"); _outf.flush()

async def run_all():
    tasks=[one(beh,pid,pr,k) for (beh,pid,pr) in sel for k in range(SAMPLES)]
    for i in range(0,len(tasks),200):
        await asyncio.gather(*tasks[i:i+200]); print(f"  {min(i+200,len(tasks))}/{len(tasks)}")

import nest_asyncio; nest_asyncio.apply()
asyncio.get_event_loop().run_until_complete(run_all())
_outf.close()
print("saved ->", OUTPUT)


In [ ]:
# === Cell 5 — extract think length + non-Latin onset ========================
import re, json
from collections import Counter
# tokenizer for a faithful think length; falls back to whitespace words.
TOK=None
try:
    from transformers import AutoTokenizer
    TOK=AutoTokenizer.from_pretrained("Qwen/Qwen3.6-35B-A3B", trust_remote_code=True)
    UNIT="tokens"
except Exception as e:
    UNIT="words"; print("tokenizer unavailable, L_cot in words:", type(e).__name__)
def length_of(t):
    if not t: return 0
    return len(TOK(t)["input_ids"]) if TOK else len(t.split())

# Non-Latin scripts: Greek, Cyrillic, Armenian, Hebrew, Arabic, Syriac, Devanagari,
# Thai, kana, CJK, Hangul, CJK-compat. Accented Latin (café, Việt) stays Latin.
FOREIGN=[(0x0370,0x03FF),(0x0400,0x04FF),(0x0500,0x052F),(0x0530,0x058F),(0x0590,0x05FF),
         (0x0600,0x06FF),(0x0700,0x074F),(0x0900,0x097F),(0x0E00,0x0E7F),
         (0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),(0xF900,0xFAFF)]
def is_foreign(ch):
    if not ch.isalpha(): return False
    o=ord(ch)
    if o<0x0250: return False               # Basic Latin + Latin-1/Extended letters
    return any(a<=o<=b for a,b in FOREIGN)
def foreign_onset(text, run=FOREIGN_RUN):
    """char index where a run of >=run non-Latin letters begins (spaces/punct don't break it)."""
    cnt=0; start=None
    for i,ch in enumerate(text):
        if is_foreign(ch):
            if cnt==0: start=i
            cnt+=1
            if cnt>=run: return start
        elif ch.isalpha():                  # a Latin letter breaks the run
            cnt=0; start=None
    return None

rows=read_jsonl(OUTPUT)
for r in rows:
    r["L_cot"]=length_of(r["reasoning"])
    on=foreign_onset(r["answer"])
    r["switched"]= on is not None
    r["onset_char"]=on
    r["onset_frac"]= (on/max(len(r["answer"]),1)) if on is not None else None
import numpy as np
print(f"{len(rows)} completions | switched: {sum(r['switched'] for r in rows)} "
      f"({100*np.mean([r['switched'] for r in rows]) if rows else 0:.0f}%) | "
      f"reasoning modes: {dict(Counter(r['mode'] for r in rows))} | L_cot unit: {UNIT}")


In [ ]:
# === Cell 6 — Part A: critical length, tipped vs not, L*, mechanism =========
import numpy as np, matplotlib.pyplot as plt
from collections import Counter
Lc=np.array([r["L_cot"] for r in rows], float); tp=np.array([r["switched"] for r in rows])
modes=Counter(r["mode"] for r in rows)

if len(rows)==0:
    print("no completions yet — run Cell 4.")
elif modes.get("no_think",0)==len(rows) or (Lc==0).all():
    print("!! no reasoning came back (all 'no_think'). The thinking flag didn't take on this provider.")
    print("   In Cell 1 set THINK_EXTRA_BODY = {'reasoning': {'effort':'high'}} (or pick a provider that")
    print("   exposes reasoning) and re-run Cell 4. Part A needs a real, varying think length.")
else:
    print(f"L_cot ({UNIT}): median={np.median(Lc):.0f}  [{np.percentile(Lc,10):.0f}, "
          f"{np.percentile(Lc,90):.0f}]  std={Lc.std():.1f}")
    if Lc.std()<1.0:
        print("!! think length barely varies — cannot act as an order parameter here.")
    if tp.any() and (~tp).any():
        lt,ln=Lc[tp],Lc[~tp]
        print(f"switched: {int(tp.sum())} ({100*tp.mean():.0f}%)")
        print(f"L_cot  switched median {np.median(lt):.0f}   not-switched median {np.median(ln):.0f}")
        rng=np.random.default_rng(0)
        diffs=[np.median(rng.choice(ln,ln.size))-np.median(rng.choice(lt,lt.size)) for _ in range(3000)]
        lo,hi=np.percentile(diffs,[2.5,97.5])
        d0=np.median(ln)-np.median(lt)
        print(f"median(not-switched) - median(switched) = {d0:.0f}  95% CI [{lo:.0f}, {hi:.0f}]")
        print("   >0 with CI excluding 0  =>  shorter thinking precedes the switch (supports L*)")
        grid=np.unique(np.percentile(Lc,np.arange(0,101,2)).astype(int))
        def youden(L):
            pred=Lc<=L; tpr=(pred&tp).sum()/max(tp.sum(),1); fpr=(pred&~tp).sum()/max((~tp).sum(),1)
            return tpr-fpr
        J=[youden(L) for L in grid]; Lstar=int(grid[int(np.argmax(J))])
        print(f"critical length  L* ≈ {Lstar} {UNIT}  (switch when think length ≤ L*; Youden J={max(J):.2f})")
        fig,ax=plt.subplots(1,2,figsize=(11,3.8))
        bins=np.linspace(0,max(np.percentile(Lc,98),1),30)
        ax[0].hist(ln,bins,alpha=.6,density=True,label="not switched")
        ax[0].hist(lt,bins,alpha=.6,density=True,label="switched")
        ax[0].axvline(Lstar,color="k",ls="--",lw=1)
        ax[0].set_xlabel(f"think length ({UNIT})"); ax[0].set_title("CoT length by outcome"); ax[0].legend()
        fr=[r["onset_frac"] for r in rows if r["switched"] and r["onset_frac"] is not None]
        ax[1].hist(fr,bins=np.linspace(0,1,25),color="C2")
        ax[1].set_xlabel("switch position in the answer (fraction)"); ax[1].set_title("where the switch starts")
        plt.tight_layout(); plt.show()
    else:
        print(f"switched fraction = {100*tp.mean():.1f}% — not both classes present to fit L*.")
        print("   If ~0%: language-switching reproduces weakly on OpenRouter — raise SAMPLES, or")
        print("   self-host the FP8 subject (the single-GPU SkyPilot YAML) and point MODEL/base-url at it.")


In [ ]:
# === Cell 7 — peek: a few switched completions ==============================
shown=0
for r in rows:
    if r["switched"] and shown<4:
        a=r["answer"]; on=r["onset_char"]
        pre=a[max(0,on-60):on].replace("\n","\\n"); post=a[on:on+80].replace("\n","\\n")
        print(f"[{r['pattern_id'][:36]}] think={r['L_cot']} {UNIT}, onset@{on} "
              f"({100*r['onset_frac']:.0f}% into answer)")
        print(f"    …{pre}  >>>  {post}…\n")
        shown+=1
if shown==0: print("no switched completions to show yet.")


### How to read it

* **`median(not-switched) − median(switched)` with a 95% CI that excludes 0** is the
  headline: positive ⇒ completions that switch had *shorter* think phases ⇒ the tip is
  a crossing below a critical CoT length `L*`. If the CI straddles 0, thinking length
  does not gate the switch (the hypothesis fails on this data).
* **`L*`** is the think-length threshold that best separates switched from not (Youden J).
* **switch-position histogram** shows *where* in the answer the language flips — near 0
  means it discharges right at the start of the answer (i.e. as soon as thinking ends).
* If **switched ≈ 0%**, the behaviour just doesn't reproduce enough on OpenRouter with
  thinking on — raise `SAMPLES`, or self-host the FP8 subject and re-point `MODEL`.
* If **all `no_think`**, the provider didn't return reasoning — change `THINK_EXTRA_BODY`.

This tests your critical-length idea end-to-end **without any GPU or draft scoring**.
